# Get credits for each track
We are using whosampled.com to get credits for each track. Credits for a track can be any combination of:
- producers
- mixers
- writers
- etc.

Each track has it's own web page and it's `html` is downloaded and saved in the input directory.

We will read each web page and get the track's credits with xpath. Some credit roles have one name credited. Others have multiple names. Each xpath is written explicitly on the `role_name` and separated for single and multiple instances of names.

Then, save to the data to a csv in the output directory. The csv is structured for the data viz bubble graphic where a long and tidy shape is required. Each bubble will need a 1:1 relationship with a role that we want to sort by (e.g. tracks by producers). In the csv, the `is_primary` column is set to **True** to handle this sorting. In the case where there are more than one producers on one track, the first producer will have their `is_primary` tag set to **True**, others will be **False**. The data viz will read this field, along with others to display the correct sorting when triggered. 

In [2]:
import os
import json
import glob
import pandas as pd
from lxml import etree, html

In [3]:
with open('consts.json', 'r') as file:
    data = json.load(file)

album = data['album']
artist = data['artist']

In [4]:
script_dir = os.path.dirname(os.path.abspath('get-credits.ipynb'))

In [5]:
files = glob.glob(os.path.join(script_dir, '..', 'albums', f'{album}', 'input', 'track_details*.html'))

In [6]:
xpath_track_name = '//nav[@class="breadcrumb-wrapper"]//ol//li//meta[@content="2"]//preceding-sibling::span//text()'

In [7]:
xpaths_credits = {
    "producer": {
        "producer": '//span[@itemprop="producer"]//text()'
    },
    "writer": {
        "writer": '//div[@class="track-credit-wrapper"]//span[@class="track-credit-title" and text()="Writers:"]//following-sibling::*//span[@itemprop="name"]//text()'
    },
    "instrumentalist": {
        "lead_vocals": '//div[@class="track-credit-wrapper"]//span[@class="track-credit-title" and text()="Lead Vocals:"]//following-sibling::*//span[@itemprop="name"]//text()',
        "background_vocals": '//div[@class="track-credit-wrapper"]//span[@class="track-credit-title" and text()="Background Vocals:"]//following-sibling::*//span[@itemprop="name"]//text()',
        "additional_vocals": '//div[@class="track-credit-wrapper"]//span[@class="track-credit-title" and text()="Additional Vocals:"]//following-sibling::*//span[@itemprop="name"]//text()',
        "guitar": '//div[@class="track-credit-wrapper"]//span[@class="track-credit-title" and text()="Guitar:"]//following-sibling::*//span[@itemprop="name"]//text()',
        "bass_guitar": '//div[@class="track-credit-wrapper"]//span[@class="track-credit-title" and text()="Bass Guitar:"]//following-sibling::*//span[@itemprop="name"]//text()',
        "drums": '//div[@class="track-credit-wrapper"]//span[@class="track-credit-title" and text()="Drums:"]//following-sibling::*//span[@itemprop="name"]//text()',
        "vibes": '//div[@class="track-credit-wrapper"]//span[@class="track-credit-title" and text()="Vibraphone:"]//following-sibling::*//span[@itemprop="name"]//text()',
        "trumpet": '//div[@class="track-credit-wrapper"]//span[@class="track-credit-title" and text()="Trumpet:"]//following-sibling::*//span[@itemprop="name"]//text()',
        "trombone": '//div[@class="track-credit-wrapper"]//span[@class="track-credit-title" and text()="Trombone:"]//following-sibling::*//span[@itemprop="name"]//text()',
        "keyboard": '//div[@class="track-credit-wrapper"]//span[@class="track-credit-title" and text()="Keyboard:"]//following-sibling::*//span[@itemprop="name"]//text()',
        "handclaps": '//div[@class="track-credit-wrapper"]//span[@class="track-credit-title" and text()="Handclaps:"]//following-sibling::*//span[@itemprop="name"]//text()',
        "vocoder": '//div[@class="track-credit-wrapper"]//span[@class="track-credit-title" and text()="Vocoder:"]//following-sibling::*//span[@itemprop="name"]//text()',
        "claves": '//div[@class="track-credit-wrapper"]//span[@class="track-credit-title" and text()="Claves:"]//following-sibling::*//span[@itemprop="name"]//text()',
        "harp": '//div[@class="track-credit-wrapper"]//span[@class="track-credit-title" and text()="Harp:"]//following-sibling::*//span[@itemprop="name"]//text()',
        "flute": '//div[@class="track-credit-wrapper"]//span[@class="track-credit-title" and text()="Flute:"]//following-sibling::*//span[@itemprop="name"]//text()',
        "piano": '//div[@class="track-credit-wrapper"]//span[@class="track-credit-title" and text()="Piano:"]//following-sibling::*//span[@itemprop="name"]//text()',
        "other_instruments": '//div[@class="track-credit-wrapper"]//span[@class="track-credit-title" and text()="Other Instruments:"]//following-sibling::*//span[@itemprop="name"]//text()'
    },
    "recorder": {
        "recorder": '//div[@class="track-credit-wrapper"]//span[@class="track-credit-title" and text()="Recorder:"]//following-sibling::*//span[@itemprop="name"]//text()',
        "recorders": '//div[@class="track-credit-wrapper"]//span[@class="track-credit-title" and text()="Recorders:"]//following-sibling::*//span[@itemprop="name"]//text()',
        "assistant_recorder": '//div[@class="track-credit-wrapper"]//span[@class="track-credit-title" and text()="Assistant Recorder:"]//following-sibling::*//span[@itemprop="name"]//text()',
        "assistant_recorders": '//div[@class="track-credit-wrapper"]//span[@class="track-credit-title" and text()="Assistant Recorders:"]//following-sibling::*//span[@itemprop="name"]//text()'
    },
    "mixer": {
        "mixer": '//div[@class="track-credit-wrapper"]//span[@class="track-credit-title" and text()="Mixer:"]//following-sibling::*//span[@itemprop="name"]//text()',
        "mixers": '//div[@class="track-credit-wrapper"]//span[@class="track-credit-title" and text()="Mixers:"]//following-sibling::*//span[@itemprop="name"]//text()',
        "assistant_mixer": '//div[@class="track-credit-wrapper"]//span[@class="track-credit-title" and text()="Assistant Mixer:"]//following-sibling::*//span[@itemprop="name"]//text()',
        "assistant_mixers": '//div[@class="track-credit-wrapper"]//span[@class="track-credit-title" and text()="Assistant Mixers:"]//following-sibling::*//span[@itemprop="name"]//text()'
    }
}

In [8]:
data = []
for fn in files:
    credits = {}
    tree = html.fromstring(open(fn).read())
    track_name = tree.xpath(xpath_track_name)

    for role, xpath_roles in xpaths_credits.items():
        for r, x in xpath_roles.items():
            role_names = tree.xpath(x)

            if len(role_names) == 1:
                credit = {
                    "track_name": track_name[0],
                    "role": role,
                    "role_detail": r,
                    "person": role_names[0],
                    "is_primary": True
                }
                data.append(credit)
            if len(role_names) > 1:
                for i, name in enumerate(role_names):
                    credit = {
                        "track_name": track_name[0],
                        "role": role,
                        "role_detail": r,
                        "person": name,
                        "is_primary": True if i == 0 else False
                    }
                    data.append(credit)

In [9]:
df = pd.DataFrame(data)
df

,track_name,role,role_detail,person,is_primary
0,The Questions,producer,producer,J Dilla,True
1,The Questions,producer,producer,James Poyser,False
2,The Questions,writer,writer,James Poyser,True
3,The Questions,writer,writer,Common,False
4,The Questions,writer,writer,Mos Def,False
...,...,...,...,...,...
182,Dooinit,instrumentalist,lead_vocals,Common,True
183,Dooinit,recorder,recorder,Todd Fairall,True
184,Dooinit,recorder,assistant_recorder,Nick Hura,True
185,Dooinit,mixer,mixer,Bob Power,True


In [10]:
output_file_path = os.path.join(script_dir, '..', 'albums', f'{album}', 'output', f'credits_{album}.csv')

In [11]:
df.to_csv(output_file_path, index=False)